In [0]:
%run ./utils

# Import

In [0]:
import pyspark.sql.functions as F

# Load

In [0]:
dbutils.widgets.text("silver_table_name","tb_chamados_hora")
silver_table_name = dbutils.widgets.get("silver_table_name")

df_data = (
    spark.table(f"{silver_schema}.{silver_table_name}")
    .distinct()
    .orderBy("id_chamado")
)

In [0]:
df_data = (
    df_data
        .select(
            F.col("id_chamado"),
            "id_cliente",
            "hora_abertura_chamado",
            "hora_inicio_atendimento",
            "hora_finalizacao_atendimento",
        )
)

# Transformação


In [0]:
# quebrar hora_inicio_atendimento em dia, mes e ano
df_data = (
    df_data
        .withColumn("dia", F.dayofmonth("hora_inicio_atendimento"))
        .withColumn("mes", F.month("hora_inicio_atendimento"))
        .withColumn("ano", F.year("hora_inicio_atendimento"))
)

In [0]:
# definir qual dia da semana é aquela data
df_data = (
    df_data
        .withColumn("dia_semana", F.dayofweek("hora_inicio_atendimento"))
)

In [0]:
# criar colunas dia util e fim de semana booleans
df_data = (
    df_data
        .withColumn("dia_util", F.when((F.col("dia_semana") >= 2) & (F.col("dia_semana") <= 6), True).otherwise(False))
        .withColumn("fim_de_semana", F.when((F.col("dia_semana") == 1) | (F.col("dia_semana") == 7), True).otherwise(False))
)

In [0]:
# criar coluna feriados, depende do id do cliente para descobrir qual estado/região ele reside para determinar o feriado
# como o comportamento altera em volta desses eventos e gera gargalos na empresa

# Surrogate Keys

In [0]:
add_sequential_key(df_data, "id_chamado")

# Salvamento